In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              precision_score, recall_score, f1_score)
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_telco_churn.csv")

In [3]:
print(df.shape)
print(df['customer_status'].value_counts())

(7043, 39)
customer_status
Stayed     4720
Churned    1869
Joined      454
Name: count, dtype: int64


In [4]:
# 1. Remove "Joined" customers — their outcome isn't determined yet
df_model = df[df['customer_status'] != 'Joined'].copy()

# 2. Create binary target: Stayed -> 0, Churned -> 1
df_model['churn_label'] = df_model['customer_status'].map({'Stayed': 0, 'Churned': 1})

# 3. Drop columns we don't need:
# - customer_status: replaced by churn_label
# - churn_category, churn_reason: leakage (only known after churn happens)
# - customer_i_d: unique identifier, no predictive value
# - zip_code, latitude, longitude: too granular, easily overfit
# - city: too many unique values, low importance, slows down training
cols_to_drop = ['customer_status', 'churn_category', 'churn_reason',
                'customer_i_d', 'zip_code', 'latitude', 'longitude', 'city']

df_model = df_model.drop(columns=cols_to_drop)

# Quick check
print(df_model.shape)
print(df_model['churn_label'].value_counts())
print(df_model.columns.tolist())

(6589, 32)
churn_label
0    4720
1    1869
Name: count, dtype: int64
['gender', 'age', 'married', 'number_of_dependents', 'number_of_referrals', 'tenure_in_months', 'offer', 'phone_service', 'avg_monthly_long_distance_charges', 'multiple_lines', 'internet_service', 'internet_type', 'avg_monthly_g_b_download', 'online_security', 'online_backup', 'device_protection_plan', 'premium_tech_support', 'streaming_t_v', 'streaming_movies', 'streaming_music', 'unlimited_data', 'contract', 'paperless_billing', 'payment_method', 'monthly_charge', 'total_charges', 'total_refunds', 'total_extra_data_charges', 'total_long_distance_charges', 'total_revenue', 'population', 'churn_label']


In [5]:
# A couple of simple, high-value ratio features — skipping the heavier ones to keep this fast
df_model['revenue_per_month'] = df_model['total_revenue'] / df_model['tenure_in_months']
df_model['refund_ratio'] = df_model['total_refunds'] / df_model['total_charges']

# Tenure buckets — simple grouping that often reveals patterns (e.g., churn spikes after contract ends)
df_model['tenure_group'] = pd.cut(
    df_model['tenure_in_months'],
    bins=[0, 12, 24, 48, 100],
    labels=['0-12mo', '12-24mo', '24-48mo', '48mo+']
)

# Clean up any inf/NaN from divisions (e.g., $0 total_charges)
ratio_cols = ['revenue_per_month', 'refund_ratio']
df_model[ratio_cols] = df_model[ratio_cols].replace([np.inf, -np.inf], 0).fillna(0)

# Quick check
print(df_model[['revenue_per_month', 'refund_ratio', 'tenure_group']].head())
print(df_model.shape)

   revenue_per_month  refund_ratio tenure_group
0         108.312222      0.000000       0-12mo
1          67.808889      0.070667       0-12mo
2         103.862500      0.000000       0-12mo
3         123.039231      0.000000      12-24mo
4          96.513333      0.000000       0-12mo
(6589, 35)


In [6]:
# List all text/category columns that need encoding
categorical_cols = ['gender', 'married', 'offer', 'phone_service', 'multiple_lines',
                     'internet_service', 'internet_type', 'online_security',
                     'online_backup', 'device_protection_plan', 'premium_tech_support',
                     'streaming_t_v', 'streaming_movies', 'streaming_music',
                     'unlimited_data', 'contract', 'paperless_billing',
                     'payment_method', 'tenure_group']

# One-hot encode directly on the dataframe — simple, no ColumnTransformer needed
df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

# Quick check
print(df_model.shape)
print(df_model.dtypes.value_counts())

(6589, 45)
bool       29
float64     9
int64       7
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = df_model.drop(columns=['churn_label'])
y = df_model['churn_label']

# Split into train/test — stratify to preserve the churn ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(5271, 44) (1318, 44)
churn_label
0    0.716373
1    0.283627
Name: proportion, dtype: float64
churn_label
0    0.716237
1    0.283763
Name: proportion, dtype: float64


In [9]:
from sklearn.preprocessing import StandardScaler

# List of numeric columns that need scaling (the boolean/dummy columns from get_dummies don't need this)
numeric_cols = ['age', 'number_of_dependents', 'number_of_referrals', 'tenure_in_months',
                'avg_monthly_long_distance_charges', 'avg_monthly_g_b_download',
                'monthly_charge', 'total_charges', 'total_refunds',
                'total_extra_data_charges', 'total_long_distance_charges',
                'total_revenue', 'population', 'revenue_per_month', 'refund_ratio']

# Fit the scaler ONLY on training data, then apply to both train and test
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(X_train_scaled[numeric_cols].describe().loc[['mean', 'std']].round(2))

      age  number_of_dependents  number_of_referrals  tenure_in_months  \
mean  0.0                  -0.0                 -0.0               0.0   
std   1.0                   1.0                  1.0               1.0   

      avg_monthly_long_distance_charges  avg_monthly_g_b_download  \
mean                               -0.0                      -0.0   
std                                 1.0                       1.0   

      monthly_charge  total_charges  total_refunds  total_extra_data_charges  \
mean            -0.0           -0.0            0.0                       0.0   
std              1.0            1.0            1.0                       1.0   

      total_long_distance_charges  total_revenue  population  \
mean                         -0.0           -0.0        -0.0   
std                           1.0            1.0         1.0   

      revenue_per_month  refund_ratio  
mean               -0.0           0.0  
std                 1.0           1.0  


In [10]:
X_train.head()

,age,number_of_dependents,number_of_referrals,tenure_in_months,avg_monthly_long_distance_charges,avg_monthly_g_b_download,monthly_charge,total_charges,total_refunds,total_extra_data_charges,...,streaming_music_Yes,unlimited_data_Yes,contract_One Year,contract_Two Year,paperless_billing_Yes,payment_method_Credit Card,payment_method_Mailed Check,tenure_group_12-24mo,tenure_group_24-48mo,tenure_group_48mo+
2370,64,0,0,43,21.64,13.0,60.00,2548.55,0.0,0,...,True,True,False,True,True,False,False,False,True,False
576,44,2,0,62,24.48,0.0,20.00,1250.10,0.0,0,...,False,False,False,True,False,False,False,False,False,True
5559,41,0,0,1,22.11,18.0,-9.00,45.80,0.0,10,...,False,False,False,False,False,False,False,False,False,False
3564,50,0,0,3,13.83,14.0,105.35,323.25,0.0,0,...,True,True,False,False,True,False,False,False,False,False
444,58,2,5,62,42.05,20.0,70.75,4263.45,0.0,0,...,False,True,True,False,True,False,False,False,False,True


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              precision_score, recall_score, f1_score)

# Reusable evaluation function
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("ROC-AUC Score:", round(roc_auc_score(y_test, y_proba), 4))
    
    return {
        'model_name': name,
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }

results = []  # scoreboard to compare models later

# Logistic Regression — using class_weight='balanced' instead of SMOTE (much faster, no oversampling)
logreg = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

logreg_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_grid = GridSearchCV(
    estimator=logreg,
    param_grid=logreg_param_grid,
    scoring='roc_auc',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

# Fit directly on the already-scaled training data
logreg_grid.fit(X_train_scaled, y_train)

print("Best parameters:", logreg_grid.best_params_)
print("Best cross-validation ROC-AUC:", round(logreg_grid.best_score_, 4))

# Evaluate on test set
best_logreg = logreg_grid.best_estimator_
logreg_results = evaluate_model("Logistic Regression", best_logreg, X_test_scaled, y_test)
results.append(logreg_results)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Best cross-validation ROC-AUC: 0.9171

===== Logistic Regression =====
              precision    recall  f1-score   support

      Stayed       0.92      0.80      0.86       944
     Churned       0.62      0.83      0.71       374

    accuracy                           0.81      1318
   macro avg       0.77      0.82      0.79      1318
weighted avg       0.84      0.81      0.82      1318

Confusion Matrix:
 [[756 188]
 [ 62 312]]
ROC-AUC Score: 0.9094


In [12]:
from xgboost import XGBClassifier

# Calculate scale_pos_weight — XGBoost's built-in imbalance handling (like class_weight, but XGBoost's own version)
# Formula: (number of negative class) / (number of positive class)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(scale_pos_weight, 2))

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight
)

xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=xgb_model,
    param_grid=xgb_param_grid,
    scoring='roc_auc',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

# Fit directly on scaled training data (XGBoost doesn't strictly need scaling, but it's already done, so no harm)
xgb_grid.fit(X_train_scaled, y_train)

print("Best parameters:", xgb_grid.best_params_)
print("Best cross-validation ROC-AUC:", round(xgb_grid.best_score_, 4))

# Evaluate on test set
best_xgb = xgb_grid.best_estimator_
xgb_results = evaluate_model("XGBoost", best_xgb, X_test_scaled, y_test)
results.append(xgb_results)

scale_pos_weight: 2.53
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best cross-validation ROC-AUC: 0.9384

===== XGBoost =====
              precision    recall  f1-score   support

      Stayed       0.92      0.85      0.89       944
     Churned       0.69      0.81      0.74       374

    accuracy                           0.84      1318
   macro avg       0.80      0.83      0.81      1318
weighted avg       0.85      0.84      0.85      1318

Confusion Matrix:
 [[805 139]
 [ 70 304]]
ROC-AUC Score: 0.9281


In [13]:
import joblib

# Save the trained XGBoost model
joblib.dump(best_xgb, 'churn_xgb_model.pkl')

# Save the scaler (needed to transform new raw numeric data the same way as training data)
joblib.dump(scaler, 'churn_scaler.pkl')

# Save the list of numeric columns and final column order (needed to rebuild the feature matrix correctly)
import json
model_metadata = {
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'final_feature_order': X_train_scaled.columns.tolist()
}
with open('churn_model_metadata.json', 'w') as f:
    json.dump(model_metadata, f)

print("Model, scaler, and metadata saved successfully.")

Model, scaler, and metadata saved successfully.


In [14]:
# Extract importance scores from the trained XGBoost model
importances = best_xgb.feature_importances_
feature_names = X_train_scaled.columns.tolist()

# Build a sorted dataframe
feat_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_importance_df.head(15))

                       feature  importance
26   internet_type_Fiber Optic    0.179646
37           contract_Two Year    0.167189
36           contract_One Year    0.104203
3             tenure_in_months    0.079123
33        streaming_movies_Yes    0.054437
1         number_of_dependents    0.048373
2          number_of_referrals    0.046814
39  payment_method_Credit Card    0.030756
16                 married_Yes    0.025438
6               monthly_charge    0.025026
5     avg_monthly_g_b_download    0.022515
28         online_security_Yes    0.020352
34         streaming_music_Yes    0.017991
0                          age    0.013986
38       paperless_billing_Yes    0.013717
